# 03 · Storage Engines — How Bytes Survive

**Part 1 · Data-Intensive Applications** · *Prerequisites: 01–02* · *Book mapping: DDIA ch. 3*

Under every data model sits a storage engine answering one question: **how do we lay out bytes
on disk so that writes are durable and reads are fast?** Today we build the two designs that
power almost every database you will ever touch — **log-structured** engines (Bitcask,
LevelDB/RocksDB, Cassandra) and **page-oriented B-trees** (Postgres, MySQL/InnoDB, most of the
relational world) — then flip from transactional to analytical workloads and build a **column
store**.

**Learning objectives.**

1. Explain why *append-only* is the fastest way to write, and what it costs.
2. Build a hash-indexed log store (Bitcask), with compaction and crash recovery.
3. Build an LSM-tree: memtable → SSTables → merge compaction → bloom filters; measure read
   amplification and watch bloom filters erase it.
4. Build a B-tree and verify its defining property: a handful of page reads regardless of size.
5. Choose between LSM and B-tree for a workload, using write/read/space amplification.
6. Explain OLTP vs OLAP, and demonstrate why analytics wants columns, not rows.

In [ ]:
import hashlib
import heapq
import json
import time
from bisect import bisect_right, insort
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from sysdes import scratch_dir, viz

viz.use_style()
rng = np.random.default_rng(seed=3)
DATA = scratch_dir("03-storage-engines")   # all files live under ~/tmp/sysdes-course/
print(f"scratch dir: {DATA}")

## 1. The world's simplest database — and its one flaw

An **append-only log**: to write, append `key,value` to the end of a file; to read, scan the
whole file and return the *last* value for the key. Appending is the cheapest thing storage
can do (sequential I/O, no seeking — recall the latency ladder: an HDD seek costs ~4 ms while
sequential bytes stream at hundreds of MB/s; even SSDs strongly prefer sequential writes).
Writes: O(1). Reads: O(n) — hopeless.

Every real engine is a answer to the follow-up question: *what extra structure (an* ***index***
*) do we maintain so reads are fast — and how much write speed do we pay for it?* This
trade-off — **indexes accelerate reads by taxing writes** — is the sun the whole storage
universe orbits.

## 2. Build 1 — Bitcask: append-only log + in-memory hash index

Design (used by Bitcask, the engine inside Riak):

- **Data on disk**: one append-only log file. Updates append a new record; old ones become
  garbage. Deletes append a **tombstone** (a special "this key is dead" record) — in an
  append-only world you cannot erase, only supersede.
- **Index in RAM**: `key → byte offset of its latest record`. Reads are one dict lookup + one
  disk seek.
- **Compaction**: periodically rewrite only the *live* records to a fresh file, reclaiming
  space from superseded ones.
- **Crash recovery**: the index is rebuildable by scanning the log.

In [ ]:
TOMBSTONE = "\x00__dead__"   # sentinel appended when a key is deleted

class Bitcask:
    """Append-only log store with an in-memory hash index (key -> offset)."""

    def __init__(self, path: Path):
        self.path = path
        self.index: dict[str, int] = {}
        self.f = open(path, "a+b")
        self._rebuild_index()           # crash recovery = replay the log

    def _rebuild_index(self) -> None:
        """Scan the log start-to-end; the LAST record for each key wins."""
        self.f.seek(0)
        offset = 0
        for line in self.f:
            key, value = json.loads(line)
            if value == TOMBSTONE:
                self.index.pop(key, None)   # a tombstone erases the key
            else:
                self.index[key] = offset
            offset += len(line)

    def put(self, key: str, value) -> None:
        self.f.seek(0, 2)                          # jump to end-of-file...
        offset = self.f.tell()
        self.f.write(json.dumps([key, value]).encode() + b"\n")  # ...and append
        self.index[key] = offset

    def delete(self, key: str) -> None:
        if key not in self.index:
            raise KeyError(key)
        self.put(key, TOMBSTONE)                   # append a tombstone...
        del self.index[key]                        # ...and forget the key

    def get(self, key: str):
        offset = self.index[key]                   # RAM lookup -> exact offset
        self.f.seek(offset)                        # one seek, one read
        _, value = json.loads(self.f.readline())
        return value

    def compact(self) -> None:
        """Rewrite only live records into a fresh log; swap it in."""
        new_path = self.path.with_suffix(".compacting")
        new_index: dict[str, int] = {}
        with open(new_path, "wb") as out:
            for key in self.index:                 # only keys that are alive
                new_index[key] = out.tell()
                out.write(json.dumps([key, self.get(key)]).encode() + b"\n")
        self.f.close()
        new_path.replace(self.path)                # atomic swap on POSIX
        self.f = open(self.path, "a+b")
        self.index = new_index

store = Bitcask(DATA / "bitcask.log")
store.put("user:1", {"name": "Ada"})
store.put("user:2", {"name": "Grace"})
store.put("user:1", {"name": "Ada Lovelace"})      # supersedes the first record
store.delete("user:2")

assert store.get("user:1") == {"name": "Ada Lovelace"}, "last write wins"
try:
    store.get("user:2")
    raise AssertionError("deleted key must not be readable")
except KeyError:
    pass
print("put / overwrite / delete OK")

In [ ]:
# Garbage accumulates: overwrite one key many times...
for i in range(2000):
    store.put("counter", i)
size_before = store.path.stat().st_size

# ...compaction keeps only the live records:
store.compact()
size_after = store.path.stat().st_size
print(f"log size: {size_before:,} B -> {size_after:,} B after compaction")
assert size_after < size_before / 100, "compaction reclaimed the superseded records"
assert store.get("counter") == 1999 and store.get("user:1") == {"name": "Ada Lovelace"}

# Crash recovery: a brand-new process (fresh instance, empty RAM) rebuilds the
# index purely from the log file.
reopened = Bitcask(store.path)
assert reopened.get("counter") == 1999, "index rebuilt from disk after 'crash'"
assert set(reopened.index) == set(store.index)
print("compaction + crash recovery OK")

Bitcask's two hard limits — and each points at the next design:

1. **Every key must fit in RAM** (the whole index is a dict). Millions of keys: fine.
   Billions: no.
2. **No range scans.** `get("user:1")` is O(1), but "all keys between user:100 and user:200"
   means scanning everything — hash indexes scatter nearby keys everywhere.

Fixing both means keeping keys **sorted**. Enter the SSTable.

## 3. Build 2 — SSTables and the LSM-tree

> **SSTable** (Sorted String Table): a file of key-value records **sorted by key**, written
> once, never modified. Sorting buys three things: range scans are sequential reads; lookups
> can use a **sparse** index (one entry per block, not per key — RAM problem solved); and
> files can be merged like in mergesort — streaming, sequential, cheap.

> **LSM-tree** (Log-Structured Merge tree): the architecture that gets sorted data to disk
> *without* random writes. Writes go to an in-RAM sorted buffer (the **memtable**); when full,
> it is flushed to disk as a new SSTable (one sequential write). Reads check memtable, then
> SSTables newest→oldest. Background **compaction** merges SSTables, dropping superseded
> records and tombstones.

In [ ]:
fig, ax = viz.system_diagram(
    nodes=[
        {"id": "w", "x": 0, "y": 1.4, "label": "writes", "kind": "client"},
        {"id": "mem", "x": 2.3, "y": 1.4, "label": "memtable\n(sorted, RAM)", "kind": "cache"},
        {"id": "s1", "x": 4.9, "y": 2.1, "label": "SSTable (new)", "kind": "store"},
        {"id": "s2", "x": 4.9, "y": 1.2, "label": "SSTable", "kind": "store"},
        {"id": "s3", "x": 4.9, "y": 0.3, "label": "SSTable (old)", "kind": "store"},
        {"id": "c", "x": 7.4, "y": 1.2, "label": "compaction\n(merge)", "kind": "service"},
        {"id": "r", "x": 2.3, "y": -0.4, "label": "reads: memtable,\nthen newest → oldest",
         "kind": "note"},
    ],
    edges=[
        {"src": "w", "dst": "mem"},
        {"src": "mem", "dst": "s1", "label": "flush when full\n(sequential write)"},
        {"src": "s2", "dst": "c", "label": ""},
        {"src": "s3", "dst": "c", "label": ""},
        {"src": "c", "dst": "s2", "label": "merged", "style": "dashed", "curve": 0.35},
    ],
    title="LSM-tree: random writes become sequential ones",
)
plt.show()

First, the bloom filter — the LSM-tree's painkiller. Then the tree itself.

### 3.1 Bloom filters

A read that misses must check *every* SSTable — expensive. A **bloom filter** is a tiny
probabilistic set kept per SSTable: it answers "is key possibly here?" with **no false
negatives** (never misses a real key) but some false positives. One filter check replaces a
disk probe for almost every irrelevant file.

Mechanics: an m-bit array + k hash functions; adding a key sets its k bits; a query ANDs them.
Theory: with n keys and b = m/n bits per key, the optimal k ≈ 0.69·b gives a false-positive
rate ≈ 0.6185^b.

In [ ]:
class BloomFilter:
    """Bit-array bloom filter with k independent hashes (salted blake2b)."""

    def __init__(self, n_keys: int, bits_per_key: int = 10):
        self.m = max(8, n_keys * bits_per_key)          # total bits
        self.k = max(1, round(0.693 * bits_per_key))    # optimal hash count
        self.bits = bytearray(self.m // 8 + 1)

    def _positions(self, key: str):
        for salt in range(self.k):
            digest = hashlib.blake2b(key.encode(), digest_size=8,
                                     salt=salt.to_bytes(8)).digest()
            yield int.from_bytes(digest) % self.m

    def add(self, key: str) -> None:
        for pos in self._positions(key):
            self.bits[pos // 8] |= 1 << (pos % 8)       # set the bit

    def might_contain(self, key: str) -> bool:
        return all(self.bits[pos // 8] & (1 << (pos % 8))
                   for pos in self._positions(key))

# Measure the false-positive rate across bits-per-key budgets:
present = [f"key{i}" for i in range(3000)]
absent = [f"nope{i}" for i in range(20000)]
budgets = [4, 6, 8, 10, 12, 14]
measured, theory = [], []
for b in budgets:
    bf = BloomFilter(len(present), bits_per_key=b)
    for key in present:
        bf.add(key)
    assert all(bf.might_contain(k) for k in present), "NO false negatives, ever"
    fp = sum(bf.might_contain(k) for k in absent) / len(absent)
    measured.append(fp)
    theory.append(0.6185 ** b)
    assert abs(fp - 0.6185 ** b) < 0.02, f"measured FP tracks theory at {b} bits/key"

fig, ax = plt.subplots()
ax.plot(budgets, theory, "--", label="theory 0.6185^bits")
ax.plot(budgets, measured, "o", label="measured")
ax.set_yscale("log")
ax.set_xlabel("bits per key")
ax.set_ylabel("false-positive rate")
ax.set_title("A byte per key buys a ~1% false-positive rate")
ax.legend()
plt.show()

### 3.2 The LSM-tree itself

Our SSTable stores sorted JSON lines with a **sparse index** (every 16th key → byte offset):
a lookup bisects the sparse index, seeks once, and scans at most one 16-line block. We count
every line scanned — that counter is our stand-in for disk I/O, and it will expose read
amplification.

In [ ]:
SPARSE_EVERY = 16

class SSTable:
    """Immutable sorted file + sparse index + bloom filter."""

    def __init__(self, path: Path, items: list[tuple[str, object]]):
        # items must be sorted by key; we write once and never modify.
        self.path = path
        self.sparse_keys: list[str] = []            # every 16th key...
        self.sparse_offsets: list[int] = []         # ...and its byte offset
        self.bloom = BloomFilter(len(items))
        self.min_key, self.max_key = items[0][0], items[-1][0]
        self.lines_scanned = 0                      # I/O counter for experiments
        with open(path, "wb") as f:
            for i, (key, value) in enumerate(items):
                if i % SPARSE_EVERY == 0:
                    self.sparse_keys.append(key)
                    self.sparse_offsets.append(f.tell())
                self.bloom.add(key)
                f.write(json.dumps([key, value]).encode() + b"\n")

    def get(self, key: str, use_bloom: bool = True):
        """Returns (found, value). Tombstones count as found (value=TOMBSTONE)."""
        if not (self.min_key <= key <= self.max_key):
            return False, None
        if use_bloom and not self.bloom.might_contain(key):
            return False, None                      # definitely absent: 0 disk reads
        # Bisect the sparse index to the last entry <= key, then scan the block.
        i = bisect_right(self.sparse_keys, key) - 1
        with open(self.path, "rb") as f:
            f.seek(self.sparse_offsets[i])
            for _ in range(SPARSE_EVERY):
                line = f.readline()
                if not line:
                    break
                self.lines_scanned += 1
                k, v = json.loads(line)
                if k == key:
                    return True, v
                if k > key:
                    break                            # sorted: we've passed it
        return False, None

    def items(self):
        with open(self.path, "rb") as f:
            for line in f:
                yield tuple(json.loads(line))


class LSMTree:
    """Memtable + stack of SSTables + merge compaction."""

    def __init__(self, directory: Path, memtable_limit: int = 400):
        self.dir = directory
        self.memtable: dict[str, object] = {}
        self.memtable_limit = memtable_limit
        self.sstables: list[SSTable] = []           # index 0 = NEWEST
        self._seq = 0

    def put(self, key: str, value) -> None:
        self.memtable[key] = value
        if len(self.memtable) >= self.memtable_limit:
            self.flush()

    def delete(self, key: str) -> None:
        self.put(key, TOMBSTONE)                    # deletes are writes too

    def flush(self) -> None:
        """Memtable -> new SSTable (ONE sequential write), newest first."""
        if not self.memtable:
            return
        items = sorted(self.memtable.items())
        self._seq += 1
        self.sstables.insert(0, SSTable(self.dir / f"seg-{self._seq:04d}.sst", items))
        self.memtable = {}

    def get(self, key: str, use_bloom: bool = True):
        if key in self.memtable:                    # newest data first
            value = self.memtable[key]
            if value == TOMBSTONE:
                raise KeyError(key)
            return value
        for table in self.sstables:                 # then newest -> oldest table
            found, value = table.get(key, use_bloom)
            if found:
                if value == TOMBSTONE:
                    raise KeyError(key)             # a tombstone shadows older versions
                return value
        raise KeyError(key)

    def compact(self) -> None:
        """K-way merge of all SSTables; newest version of each key wins;
        tombstones and superseded versions are dropped."""
        self.flush()
        merged: list[tuple[str, object]] = []

        # Tag each record with its table's age so that when the same key appears
        # in several tables, the merge yields the NEWEST version (lowest age) first.
        def tagged(table: SSTable, age: int):
            for k, v in table.items():
                yield (k, age, v)

        streams = [tagged(t, age) for age, t in enumerate(self.sstables)]
        last_key = None
        for k, age, v in heapq.merge(*streams):
            if k == last_key:
                continue                            # an older version: superseded
            last_key = k
            if v != TOMBSTONE:                      # tombstones die at full compaction
                merged.append((k, v))
        for t in self.sstables:
            t.path.unlink()                         # remove the old segment files
        self._seq += 1
        self.sstables = ([SSTable(self.dir / f"seg-{self._seq:04d}.sst", merged)]
                         if merged else [])

    def range(self, lo: str, hi: str) -> list[tuple[str, object]]:
        """All live (key, value) with lo <= key <= hi - the query Bitcask can't do."""
        result: dict[str, object] = {}
        for source in reversed([self.memtable.items()]
                               + [t.items() for t in self.sstables]):
            for k, v in source:                     # oldest first; newer overwrite
                if lo <= k <= hi:
                    result[k] = v
        return sorted((k, v) for k, v in result.items() if v != TOMBSTONE)

lsm = LSMTree(scratch_dir("03-storage-engines/lsm"))
print("LSM-tree ready")

**Property test.** Before trusting any storage engine, torture it: thousands of random puts,
overwrites, and deletes, mirrored against a plain dict as the *reference model*. The engine
must agree with the model on every key — present, overwritten, deleted, or never written —
and again after compaction.

In [ ]:
reference: dict[str, object] = {}
keys = [f"user:{i:05d}" for i in range(1500)]
for _ in range(6000):
    key = keys[int(rng.integers(len(keys)))]
    op = rng.random()
    if op < 0.70 or key not in reference:           # 70% put / overwrite
        value = int(rng.integers(1_000_000))
        lsm.put(key, value)
        reference[key] = value
    else:                                           # 30% delete
        lsm.delete(key)
        del reference[key]

def agrees_with_reference(tree) -> None:
    for key in keys:                                # every key ever touched
        if key in reference:
            assert tree.get(key) == reference[key], f"wrong value for {key}"
        else:
            try:
                tree.get(key)
                raise AssertionError(f"{key} should be deleted/absent")
            except KeyError:
                pass

agrees_with_reference(lsm)
n_segments_before = len(lsm.sstables) + 1           # + memtable
lsm.compact()
agrees_with_reference(lsm)                          # compaction must change nothing

expected = sorted((k, v) for k, v in reference.items() if "user:003" <= k <= "user:004")
assert lsm.range("user:003", "user:004") == expected, "sorted range scan works"
print(f"LSM agrees with reference model across 6000 random ops "
      f"({n_segments_before} segments -> {len(lsm.sstables)} after compaction); "
      f"range scans OK")

**Read amplification, measured.** A lookup may probe many SSTables — that is **read
amplification**. Two cures: compaction (fewer tables) and bloom filters (skip tables that
can't contain the key). Let's quantify the bloom filter's effect on lookups of *absent* keys
(the worst case — every table must be consulted):

In [ ]:
# Rebuild a fragmented tree (many segments, no compaction):
frag = LSMTree(scratch_dir("03-storage-engines/frag"))
for key in keys:
    frag.put(key, 1)
for key in keys[::3]:
    frag.put(key, 2)                                # overwrites -> more segments
frag.flush()

def lines_scanned(tree, probe_keys, use_bloom) -> int:
    for t in tree.sstables:
        t.lines_scanned = 0
    for key in probe_keys:
        try:
            tree.get(key, use_bloom=use_bloom)
        except KeyError:
            pass
    return sum(t.lines_scanned for t in tree.sstables)

# Absent keys chosen INSIDE the tables' key range, so the min/max shortcut
# can't reject them - only the bloom filter (or a block scan) can.
absent_keys = [f"user:{i:05d}x" for i in range(300)]
without_bloom = lines_scanned(frag, absent_keys, use_bloom=False)
with_bloom = lines_scanned(frag, absent_keys, use_bloom=True)

print(f"{len(frag.sstables)} segments; 300 absent-key lookups scanned "
      f"{without_bloom:,} lines without bloom vs {with_bloom:,} with bloom")
assert without_bloom > 20 * max(with_bloom, 1), "bloom filters erase miss-lookup I/O"

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(["no bloom", "bloom"], [without_bloom, with_bloom], color=["#c62828", "#2e7d32"])
ax.set_ylabel("lines read from disk")
ax.set_title("Read amplification on absent keys, with and without bloom filters")
plt.show()

## 4. Build 3 — the B-tree

> **B-tree**: keep keys sorted inside fixed-size **pages** (~4 KB; here, `ORDER` keys per
> node) arranged as a shallow tree. A lookup reads one page per level; inserts go *in place*
> into the right leaf, **splitting** full pages upward. With a branching factor of hundreds,
> billions of keys fit in 3–4 levels — 3–4 page reads per lookup, forever.

Where the LSM-tree *never* modifies files, the B-tree *always* modifies in place. That is the
whole philosophical split — and why B-trees need a **write-ahead log (WAL)**: a crash halfway
through a page split must be repairable, so every change is appended to a log *before* the
page is touched. (The log sneaks back in even here.)

In [ ]:
ORDER = 32   # max keys per node; real engines fit ~100-500 keys in a 4KB page

class BNode:
    __slots__ = ("keys", "values", "children")
    def __init__(self):
        self.keys: list = []
        self.values: list = []
        self.children: list["BNode"] = []   # empty -> leaf
    @property
    def is_leaf(self):
        return not self.children

class BTree:
    """Insert + search + range, with page splits and a page-read counter."""

    def __init__(self):
        self.root = BNode()
        self.page_reads = 0

    def search(self, key):
        node = self.root
        while True:
            self.page_reads += 1                   # one page fetched per level
            i = bisect_right(node.keys, key) - 1
            if i >= 0 and node.keys[i] == key:
                return node.values[i]
            if node.is_leaf:
                raise KeyError(key)
            node = node.children[bisect_right(node.keys, key)]

    def insert(self, key, value):
        root = self.root
        if len(root.keys) == ORDER:                # root full: grow a new root
            self.root = BNode()
            self.root.children.append(root)
            self._split_child(self.root, 0)
        self._insert_nonfull(self.root, key, value)

    def _split_child(self, parent: BNode, i: int) -> None:
        """Split parent's full child i into two; middle key moves up."""
        full = parent.children[i]
        mid = ORDER // 2
        right = BNode()
        right.keys, right.values = full.keys[mid + 1:], full.values[mid + 1:]
        right.children = full.children[mid + 1:]
        parent.keys.insert(i, full.keys[mid])
        parent.values.insert(i, full.values[mid])
        parent.children.insert(i + 1, right)
        full.keys, full.values = full.keys[:mid], full.values[:mid]
        full.children = full.children[:mid + 1]

    def _insert_nonfull(self, node: BNode, key, value) -> None:
        i = bisect_right(node.keys, key) - 1
        if i >= 0 and node.keys[i] == key:         # overwrite in place
            node.values[i] = value
            return
        if node.is_leaf:
            insort(node.keys, key)
            node.values.insert(node.keys.index(key), value)
            return
        child_i = bisect_right(node.keys, key)
        if len(node.children[child_i].keys) == ORDER:
            self._split_child(node, child_i)       # make room BEFORE descending
            if key > node.keys[child_i]:
                child_i += 1
            elif key == node.keys[child_i]:
                node.values[child_i] = value
                return
        self._insert_nonfull(node.children[child_i], key, value)

    def height(self) -> int:
        node, h = self.root, 1
        while not node.is_leaf:
            node, h = node.children[0], h + 1
        return h

btree = BTree()
model: dict[int, int] = {}
for _ in range(60_000):
    k = int(rng.integers(0, 200_000))
    v = int(rng.integers(1_000_000))
    btree.insert(k, v)
    model[k] = v

# Correctness against the reference model:
for k in list(model)[::37]:
    assert btree.search(k) == model[k]
try:
    btree.search(-1)
    raise AssertionError("absent key must raise")
except KeyError:
    pass

# THE B-tree property: tens of thousands of keys, a handful of page reads.
btree.page_reads = 0
btree.search(list(model)[123])
print(f"{len(model):,} keys -> height {btree.height()}, "
      f"{btree.page_reads} page reads per lookup")
assert btree.height() <= 4, "logarithmic height with a real branching factor"
assert btree.page_reads <= 4

### LSM vs B-tree: the three amplifications

| | **LSM-tree** | **B-tree** |
|---|---|---|
| Write path | append to memtable → sequential flush | WAL append + in-place page write |
| **Write amplification** (bytes written per byte of data) | compaction rewrites each record at every level (~10–30×) | page rewritten for every small change (a 100-byte row dirties a 4 KB page) |
| **Read amplification** | check memtable + several tables (blooms mitigate) | one page per level (~3–4), very predictable |
| **Space amplification** | superseded versions live until compaction | pages sit partly empty (fragmentation) |
| Range scans | merge across tables (good; sorted) | walk the leaves (excellent) |
| Sweet spot | write-heavy, recent-data reads (metrics, events, queues) | read-heavy, point + range lookups, predictable latency (classic OLTP) |

Neither dominates: **LSM turns random writes into sequential ones and pays at read + compaction
time; B-tree pays at write time and reads are flat forever.** Compaction is the LSM's ongoing
mortgage — fall behind under write bursts and reads slow down and disks fill (a classic
production incident).

## 5. OLTP vs OLAP — and why analytics wants columns

> **OLTP** (online transaction processing): many tiny reads/writes of individual records by
> key — the engines above. **OLAP** (online analytical processing): few, huge queries that
> scan millions of records but touch a *handful of columns* ("average order value by country
> by month").

Row storage makes OLAP cry: scanning "just the `amount` column" still drags every row's every
byte through memory. A **column store** keeps each column in its own contiguous array — a
query reads *only* the columns it mentions, and per-column data compresses beautifully.

In [ ]:
N = 400_000
events_cols = {                       # COLUMNAR: one contiguous array per column
    "day": rng.integers(0, 365, N).astype(np.int16),
    "country": rng.integers(0, 40, N).astype(np.uint8),   # dictionary-encoded
    "amount": np.round(rng.exponential(30, N), 2),
}
COUNTRIES = [f"C{i:02d}" for i in range(40)]              # the dictionary
events_rows = list(zip(events_cols["day"].tolist(),
                       events_cols["country"].tolist(),
                       events_cols["amount"].tolist()))   # ROW store: list of tuples

def revenue_by_day_rows(country: int) -> np.ndarray:
    """Row-at-a-time scan: touch every field of every row."""
    total = np.zeros(365)
    for day, c, amount in events_rows:
        if c == country:
            total[day] += amount
    return total

def revenue_by_day_cols(country: int) -> np.ndarray:
    """Columnar scan: touch ONLY the three referenced columns, vectorized."""
    mask = events_cols["country"] == country
    return np.bincount(events_cols["day"][mask],
                       weights=events_cols["amount"][mask], minlength=365)

t0 = time.perf_counter(); by_rows = revenue_by_day_rows(7); t_rows = time.perf_counter() - t0
t0 = time.perf_counter(); by_cols = revenue_by_day_cols(7); t_cols = time.perf_counter() - t0

assert np.allclose(by_rows, by_cols), "identical answers"
print(f"revenue-by-day over {N:,} events: rows {t_rows*1e3:.0f} ms, "
      f"columns {t_cols*1e3:.1f} ms -> {t_rows/t_cols:.0f}x faster")
assert t_cols * 5 < t_rows, "columnar scans win big on analytics"

Compression is the second half of the story. Within one column, values are self-similar:
low-cardinality strings become small ints via a **dictionary** (`country` above is already
one — 1 byte instead of a 3+ char string), and *sorted* columns collapse under **run-length
encoding** (RLE: store `(value, run length)` pairs instead of repeats):

In [ ]:
def run_length_encode(arr: np.ndarray) -> list[tuple[int, int]]:
    """[(value, run_length), ...] - tiny when equal values sit together."""
    runs, start = [], 0
    for i in range(1, len(arr) + 1):
        if i == len(arr) or arr[i] != arr[start]:
            runs.append((int(arr[start]), i - start))
            start = i
    return runs

country = events_cols["country"]
raw_strings = sum(len(COUNTRIES[c]) for c in country[:50_000])        # naive strings
dict_encoded = country[:50_000].nbytes                                # 1B per value
rle_unsorted = len(run_length_encode(country[:50_000])) * 3           # ~3B per run
rle_sorted = len(run_length_encode(np.sort(country[:50_000]))) * 3

print(f"50k country values: strings {raw_strings:,} B | dictionary {dict_encoded:,} B | "
      f"RLE unsorted {rle_unsorted:,} B | RLE sorted {rle_sorted:,} B")
assert dict_encoded < raw_strings / 2, "dictionary encoding shrinks low-cardinality columns"
assert rle_sorted < dict_encoded / 100, "sort order is a compression algorithm"
assert rle_unsorted > rle_sorted * 100, "RLE without sorting buys almost nothing"

That last pair of asserts is a design lesson wearing a compression costume: **choosing the
sort order of a column file is choosing which queries are fast and which columns are tiny.**
Parquet, the columnar file format the ML world runs on, is exactly these ideas
industrialized — we use it hands-on in nb 14. And because column files are write-once sorted
runs that get merged... they are SSTables in a suit. The ideas rhyme all the way down.

## 6. Exercises

1. **Tombstone necessity.** In the LSM-tree, why must a delete write a tombstone rather than
   simply removing the key from the memtable? Construct the exact bug that would occur
   without one (which cell above would fail?), and explain what finally reclaims the space.
2. **Write amplification estimate.** A leveled LSM has levels of 10× increasing size. A record
   is eventually rewritten by compaction once per level. With 5 levels, roughly how many bytes
   hit disk per byte of application data? What workload makes this tax worthwhile anyway?
3. **Crash mid-split.** Describe concretely what state a B-tree file could be left in if the
   process dies halfway through a page split, and how a write-ahead log repairs it on restart.
4. **Engine choice.** Workload A: 200k writes/s of sensor readings, reads are dashboards over
   the last hour. Workload B: bank ledger, 2k writes/s, heavy point reads + range scans over
   account history, strict p99. Choose an engine for each and name the tuning knob you would
   watch first.
5. **Sort order as design.** An events table is queried 90% as "all events for user X in time
   range" and 10% as "all events of type T yesterday". Choose the sort key of the columnar
   files, and explain what the 10% query costs and one mitigation.

### Solutions

**1.** `get` falls through to older SSTables when the memtable/newer tables lack a key. If
delete merely removed the memtable entry, an older SSTable would still hold a stale value and
`get` would happily return it — a deleted key resurrects (the property test's deleted-key
check would fail). The tombstone *shadows* older versions; the space is reclaimed at full
compaction, when the tombstone has out-lived every older version and can itself be dropped.

**2.** ~1 write on flush + ~1 rewrite per level ≈ 6× just for structure, and each level-merge
also rewrites its neighbors' data — real systems measure 10–30×. Worth it when writes are
random-keyed and heavy: the alternative (B-tree) would turn each into a random 4 KB page
write, and sequential-vs-random can be a 100× device-level difference (nb 01's ladder).

**3.** Mid-split, the file can hold: a new right page written, the parent not yet pointing at
it (orphan page — leaked space), or worse, the parent updated but the split pages half-copied
(dangling/corrupt pointers). WAL repair: every intended page change was appended to the log
first; on restart, replay the log's complete records and ignore its torn tail — pages return
to a state that matches a prefix of committed operations. (Notebook 07 builds on exactly this
idea for atomicity.)

**4.** A: LSM — sequential ingest is the whole game; watch **compaction backlog** (if merges
fall behind, reads and disk usage bloat). B: B-tree — flat, predictable read latency and
mature range scans; watch **page cache hit rate / checkpoint & WAL fsync latency**, which is
where its p99 lives.

**5.** Sort by `(user_id, timestamp)`: the 90% query reads one contiguous run (and RLE makes
`user_id` nearly free). The 10% query now scans everything (type is scattered); mitigations:
a secondary copy sorted by `(type, timestamp)` — storage is cheap, this is just
denormalization again (nb 02) — or per-file min/max metadata on `type` to prune files
(exactly what Parquet does; nb 14).

## 7. Takeaways

- Storage design = **choosing your amplification**: indexes tax writes to bless reads; LSM
  pays in compaction and multi-table reads; B-trees pay in page writes; column stores pay
  write-time sorting to make scans nearly free.
- **Appending is sacred**: logs give cheap writes and crash recovery (Bitcask rebuilds its
  RAM index; B-trees keep a WAL; the log reappears as the backbone of replication (nb 05)
  and streaming (nb 11)).
- **Sorting is the master key**: sparse indexes, mergeable files, range scans, and compression
  all fall out of keeping keys in order.
- **Bloom filters**: one byte per key ≈ 1% false positives ≈ absent-key reads for free.
- Know your workload: OLTP wants trees; OLAP wants columns; picking the sort key is picking
  your fast queries.

**Further reading.** DDIA ch. 3; O'Neil et al., *The Log-Structured Merge-Tree* (1996);
the RocksDB tuning guide (amplifications, live); Abadi et al., *Column-Stores vs Row-Stores*
(SIGMOD 2008).

**Next:** [04 · Encoding & evolution](04-encoding-evolution.ipynb) — the bytes are stored;
now they must travel between programs *and between versions of your code*, without anything
breaking.